# Flagship Registry — code source complet
Chaque cellule correspond à un fichier. Lancer avec `streamlit run flagship_app.py`.


In [ ]:
# === flagship_app.py ===
import streamlit as st
from db import init_db, q1
from pages import (
    render_login_page,
    render_list_page,
    render_flagship_detail_page,
    render_flagship_form_page,
    render_usecase_detail_page,
    render_usecase_form_page,
    render_country_form_page,
    render_dashboard_page,
    render_users_admin_page,
    render_audit_history_page,
)

st.set_page_config(page_title="Flagship Registry", layout="centered")

init_db()

# Session state
if "logged_in" not in st.session_state:
    st.session_state.logged_in = False
if "filter_countries" not in st.session_state:
    st.session_state.filter_countries = []
if "filter_divisions" not in st.session_state:
    st.session_state.filter_divisions = []
if "filter_statuses" not in st.session_state:
    st.session_state.filter_statuses = []

if not st.session_state.logged_in:
    render_login_page()
    st.stop()

is_admin = st.session_state.get("is_admin", False)

# ── Sidebar nav ───────────────────────────────────────────────────────────────
with st.sidebar:
    role = "👑 Admin" if is_admin else "👤 Utilisateur"
    st.write(f"{role} — **{st.session_state.username}**")
    if st.button("🏠 Accueil"):
        st.session_state.page = "list"
        st.rerun()

    if is_admin:
        if st.button("📈 Dashboard"):
            st.session_state.page = "dashboard"
            st.rerun()
        if st.button("⚙️ Paramètres Habilitations"):
            st.session_state.page = "users_admin"
            st.rerun()
        if st.button("📜 Historique des Actions"):
            st.session_state.page = "audit_history"
            st.rerun()

    if st.session_state.get("page") in ("flagship_detail", "flagship_form", "usecase_detail", "usecase_form", "country_form"):
        fid = st.session_state.get("current_flagship_id")
        f = q1("SELECT name FROM flagships WHERE id=?", fid) if fid else None
        if f and st.button(f"↩ {f['name']}"):
            st.session_state.page = "flagship_detail"
            st.rerun()
    if st.session_state.get("page") in ("usecase_detail", "usecase_form", "country_form"):
        ucid = st.session_state.get("current_usecase_id")
        uc = q1("SELECT name FROM use_cases WHERE id=?", ucid) if ucid else None
        if uc and st.button(f"↩ {uc['name']}"):
            st.session_state.page = "usecase_detail"
            st.rerun()
    st.divider()
    if st.button("🚪 Déconnexion"):
        st.session_state.clear()
        st.rerun()

# ── Routing ───────────────────────────────────────────────────────────────────
page = st.session_state.get("page", "list")

if page == "list":
    render_list_page()
elif page == "flagship_detail":
    render_flagship_detail_page()
elif page == "flagship_form":
    render_flagship_form_page()
elif page == "usecase_detail":
    render_usecase_detail_page()
elif page == "usecase_form":
    render_usecase_form_page()
elif page == "country_form":
    render_country_form_page()
elif page == "dashboard":
    render_dashboard_page()
elif page == "users_admin":
    render_users_admin_page()
elif page == "audit_history":
    render_audit_history_page()


In [ ]:
# === db.py ===
import sqlite3
from pathlib import Path

DB_PATH = Path(__file__).parent / "flagship.db"

# Natures de l'IA value (Net Banking Impact, Gross Cost Savings, Cost Avoidance,
# Cost of Risk, RWA Savings)
VALUE_NATURES = [
    "Net Banking Impact",
    "Gross Cost Savings",
    "Cost Avoidance",
    "Cost of Risk",
    "RWA Savings",
]


def get_db():
    conn = sqlite3.connect(str(DB_PATH), check_same_thread=False)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    return conn


def init_db():
    from auth import hash_pw  # lazy import to avoid circular dependency
    conn = get_db()
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        username TEXT UNIQUE NOT NULL,
        password_hash TEXT NOT NULL,
        is_admin INTEGER NOT NULL DEFAULT 0
    );
    CREATE TABLE IF NOT EXISTS flagships (
        id TEXT PRIMARY KEY, rank INTEGER, name TEXT NOT NULL,
        description TEXT, sponsor TEXT, owner TEXT,
        division TEXT, subdivision TEXT, poc TEXT, analyst TEXT
    );
    CREATE TABLE IF NOT EXISTS use_cases (
        id TEXT PRIMARY KEY, flagship_id TEXT NOT NULL,
        name TEXT NOT NULL, description TEXT,
        pole TEXT, entity TEXT, ai_act TEXT, technology TEXT,
        platform TEXT, brick TEXT, domain TEXT, category TEXT,
        sourcing TEXT, contributors TEXT, pilot TEXT, itsvc TEXT,
        date_itsvc TEXT, gate TEXT, date_gate TEXT
    );
    CREATE TABLE IF NOT EXISTS countries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        use_case_id TEXT NOT NULL, country TEXT NOT NULL,
        status TEXT, prod_date TEXT,
        build REAL DEFAULT 0,
        measure TEXT, rollout TEXT, mrr_cat TEXT, mrr TEXT, owner TEXT,
        UNIQUE(use_case_id, country)
    );
    CREATE TABLE IF NOT EXISTS value_entries (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        use_case_id TEXT NOT NULL,
        country TEXT NOT NULL,
        year INTEGER NOT NULL,
        nature TEXT NOT NULL,
        amount REAL NOT NULL DEFAULT 0,
        UNIQUE(use_case_id, country, year, nature)
    );
    CREATE TABLE IF NOT EXISTS user_countries (
        username TEXT NOT NULL,
        country_id INTEGER NOT NULL,
        PRIMARY KEY(username, country_id),
        FOREIGN KEY(country_id) REFERENCES countries(id) ON DELETE CASCADE
    );
    CREATE TABLE IF NOT EXISTS audit_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
        username TEXT NOT NULL,
        action TEXT NOT NULL,
        table_name TEXT NOT NULL,
        record_id TEXT NOT NULL,
        details TEXT NOT NULL
    );
    """)
    conn.commit()

    if conn.execute("SELECT COUNT(*) FROM users").fetchone()[0] == 0:
        conn.execute("INSERT INTO users (username, password_hash, is_admin) VALUES (?,?,1)",
                     ("admin", hash_pw("adminflagship2026")))
        conn.commit()

    if conn.execute("SELECT COUNT(*) FROM flagships").fetchone()[0] == 0:
        seed(conn)

    conn.close()


def seed(conn):
    flagships = [
        ("F01", 1, "KYC Acceleration", "Automated KYC document review and risk scoring.", "Group Compliance", "M. Lefevre", "GBIS", "Financial Crime", "A. Moreau", "C. Dubois"),
        ("F02", 2, "Credit Decision Copilot", "Generative AI copilot for credit analysts.", "GLBA", "J. Bernard", "Corporate & Investment Banking", "Credit Risk", "P. Lambert", "L. Garnier"),
        ("F03", 3, "Markets AI Trading Assistant", "Real-time AI assistant for FX and rates traders.", "Global Markets", "S. Nakamura", "Global Markets", "FX & Rates", "R. Chen", "A. Petit"),
        ("F04", 4, "Retail Customer 360", "Unified customer view and next-best-action engine.", "Retail Banking", "C. Martin", "Retail Banking", "Customer Experience", "V. Roux", "M. Faure"),
        ("F05", 5, "AML Transaction Monitoring", "Next-gen anti-money laundering alerting.", "Group Compliance", "A. Kovac", "GBIS", "AML", "N. Olsen", "T. Weiss"),
        ("F06", 6, "IT Ops AI", "AIOps platform for incident prediction and routing.", "Group IT", "D. Klein", "Group IT", "Infrastructure", "E. Tran", "O. Mendes"),
        ("F07", 7, "ESG Intelligence", "AI-driven ESG scoring and controversy detection.", "Sustainability", "I. Aliyev", "Group Strategy", "Sustainability", "B. Costa", "H. Park"),
        ("F08", 8, "Document Intelligence Hub", "Centralized document understanding service.", "Operations", "F. Boucher", "GBIS", "Operations", "K. Iyer", "D. Souza"),
        ("F09", 9, "Fraud Shield", "Real-time fraud detection across cards and transfers.", "Risk", "R. Hoffmann", "Retail Banking", "Fraud", "L. Yamada", "P. Silva"),
        ("F10", 10, "HR Talent AI", "AI-assisted CV screening and internal mobility.", "Group HR", "V. Almeida", "Group HR", "Talent", "S. Kone", "E. Ozturk"),
        ("F11", 11, "Treasury & ALM Optimizer", "AI-driven liquidity forecasting.", "Treasury", "M. Eriksson", "Finance", "ALM", "T. Andersson", "J. Lindqvist"),
        ("F12", 12, "Wealth Advisor Copilot", "Generative AI assistant for private bankers.", "Private Banking", "L. Fontaine", "Wealth Management", "Private Banking", "M. Yilmaz", "N. Rahman"),
        ("F13", 13, "Insurance Claims AI", "Automated claims triage and fraud detection.", "Insurance", "P. Novak", "Insurance", "Claims", "A. Vargas", "C. Romero"),
        ("F14", 14, "Regulatory Reporting AI", "Automated extraction of regulatory reports.", "Finance", "B. Vargas", "Finance", "Regulatory Reporting", "M. Diallo", "V. Ahmed"),
        ("F15", 15, "Cybersecurity AI", "AI-powered SOC: threat hunting, phishing detection.", "Group Security", "T. Berg", "Group IT", "Cybersecurity", "H. Mensah", "O. Karimov"),
        ("F16", 16, "Climate Stress Tests AI", "AI models for climate scenario analysis.", "Risk", "C. Iversen", "Risk", "Climate Risk", "A. Petrov", "L. Singh"),
        ("F17", 17, "Customer Service Voice AI", "Voice AI for inbound retail call centers.", "Retail Banking", "I. Renaud", "Retail Banking", "Customer Service", "D. Singh", "R. Costa"),
        ("F18", 18, "Data Governance AI", "AI for metadata enrichment and data quality.", "CDO Office", "S. Andrade", "Group Data", "Data Governance", "F. Costa", "M. Khan"),
    ]
    conn.executemany("INSERT OR IGNORE INTO flagships (id, rank, name, description, sponsor, owner, division, subdivision, poc, analyst) VALUES (?,?,?,?,?,?,?,?,?,?)", flagships)

    use_cases = [
        ("U01F01", "F01", "Document Extraction Pipeline", "Extract KYC fields from documents.", "Operations", "SG Group", "High Risk", "LLM + OCR", "Azure ML", "Document AI", "NLP", "Information Extraction", "Hybrid", "AI4Compliance", "France", "ITSVC-1042", "2024-03-15", "Gate 4", "2024-09-20"),
        ("U02F01", "F01", "Risk Scoring Engine", "PEP/sanction screening and risk scoring.", "Compliance", "SG Group", "High Risk", "Gradient Boosting", "Databricks", "Predictive AI", "Tabular ML", "Classification", "In-house", "Risk Analytics", "France", "ITSVC-1078", "2024-06-10", "Gate 3", "2025-01-12"),
        ("U01F02", "F02", "Credit Memo Summarization", "Summarize credit memos into key risk factors.", "Front Office", "GLBA", "Limited Risk", "LLM (GPT-4)", "Internal LLM Gateway", "Generative AI", "NLP", "Summarization", "Vendor", "GenAI Lab", "France", "ITSVC-1130", "2024-09-01", "Gate 4", "2025-02-28"),
        ("U01F03", "F03", "News Triage Engine", "Cluster and score market-moving news.", "Trading Floor", "GLBA Markets", "Limited Risk", "Transformers", "AWS SageMaker", "NLP Pipeline", "NLP", "Information Retrieval", "In-house", "Markets AI Squad", "France", "ITSVC-1201", "2024-11-12", "Gate 4", "2025-04-08"),
        ("U01F04", "F04", "Next-Best-Action Recommender", "Personalized product recommendations.", "Retail", "SG Retail FR", "Limited Risk", "Two-Tower DL", "GCP Vertex AI", "Recommendation", "Recsys", "Personalization", "In-house", "Personalization Lab", "France", "ITSVC-1156", "2024-08-20", "Gate 4", "2025-01-30"),
        ("U01F05", "F05", "Graph-Based Alert Triage", "GNN-based suspicious activity scoring.", "Compliance", "SG Group", "High Risk", "Graph Neural Networks", "Databricks", "Predictive AI", "Graph ML", "Anomaly Detection", "In-house", "Graph AI Team", "France", "ITSVC-1210", "2024-12-01", "Gate 3", "2025-03-22"),
        ("U01F06", "F06", "Incident Auto-Routing", "Classify and route IT tickets.", "IT Ops", "SG Group", "Minimal Risk", "BERT Classifier", "Azure ML", "NLP Classifier", "NLP", "Classification", "In-house", "AIOps Squad", "France", "ITSVC-1067", "2024-05-10", "Gate 4", "2024-10-15"),
    ]
    conn.executemany("INSERT OR IGNORE INTO use_cases (id, flagship_id, name, description, pole, entity, ai_act, technology, platform, brick, domain, category, sourcing, contributors, pilot, itsvc, date_itsvc, gate, date_gate) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", use_cases)

    # build au niveau pays (coût de build one-shot)
    countries_data = [
        ("U01F01", "France", "Industrialized", "2024-10-01", 1200, "Measured", "Local", "Operational Efficiency", "MRR-FR-2024-018", "M. Lefevre"),
        ("U01F01", "Italy", "Pilot", "2026-08-15", 400, "Estimated", "Local", "Operational Efficiency", "MRR-IT-2025-003", "G. Rossi"),
        ("U02F01", "France", "Pilot", "2025-02-01", 600, "Estimated", "Local", "Risk Management", "MRR-FR-2025-006", "M. Lefevre"),
        ("U01F02", "France", "Industrialized", "2025-03-01", 900, "Measured", "Global", "Productivity", "MRR-FR-2025-011", "J. Bernard"),
        ("U01F02", "Luxembourg", "Pilot", "2026-09-15", 300, "Estimated", "Local", "Productivity", "MRR-LU-2025-002", "F. Schmit"),
        ("U01F03", "France", "Industrialized", "2025-04-10", 1500, "Measured", "Global", "Revenue", "MRR-FR-2025-019", "S. Nakamura"),
        ("U01F04", "France", "Industrialized", "2025-02-15", 800, "Measured", "Local", "Revenue", "MRR-FR-2025-009", "C. Martin"),
        ("U01F05", "France", "Pilot", "2026-06-01", 700, "Estimated", "Local", "Risk Management", "MRR-FR-2025-014", "A. Kovac"),
        ("U01F06", "France", "Industrialized", "2024-11-01", 500, "Measured", "Global", "Operational Efficiency", "MRR-FR-2024-029", "D. Klein"),
    ]
    conn.executemany("INSERT OR IGNORE INTO countries (use_case_id,country,status,prod_date,build,measure,rollout,mrr_cat,mrr,owner) VALUES (?,?,?,?,?,?,?,?,?,?)", countries_data)

    # IA value par année et par nature (use_case_id, country, year, nature, amount)
    value_entries = [
        ("U01F01", "France", 2025, "Gross Cost Savings", 2200),
        ("U01F01", "France", 2025, "Cost Avoidance", 600),
        ("U01F01", "France", 2026, "Gross Cost Savings", 2500),
        ("U01F01", "Italy", 2026, "Gross Cost Savings", 700),
        ("U01F01", "Italy", 2026, "Cost Avoidance", 200),
        ("U02F01", "France", 2025, "Cost of Risk", 1500),
        ("U01F02", "France", 2025, "Net Banking Impact", 1800),
        ("U01F02", "France", 2025, "Gross Cost Savings", 1400),
        ("U01F02", "Luxembourg", 2026, "Net Banking Impact", 400),
        ("U01F02", "Luxembourg", 2026, "Gross Cost Savings", 200),
        ("U01F03", "France", 2025, "Net Banking Impact", 4500),
        ("U01F04", "France", 2025, "Net Banking Impact", 3400),
        ("U01F05", "France", 2026, "Cost of Risk", 2100),
        ("U01F06", "France", 2025, "Gross Cost Savings", 1400),
    ]
    conn.executemany("INSERT OR IGNORE INTO value_entries (use_case_id,country,year,nature,amount) VALUES (?,?,?,?,?)", value_entries)
    conn.commit()


def q(sql, *args):
    conn = get_db()
    rows = conn.execute(sql, args).fetchall()
    conn.close()
    return [dict(r) for r in rows]


def q1(sql, *args):
    conn = get_db()
    r = conn.execute(sql, args).fetchone()
    conn.close()
    return dict(r) if r else None


def run(sql, *args):
    conn = get_db()
    conn.execute(sql, args)
    conn.commit()
    conn.close()


def next_f_id():
    r = q1("SELECT MAX(CAST(SUBSTR(id,2) AS INT)) n FROM flagships WHERE id LIKE 'F%'")
    return f"F{(r['n'] or 0)+1:02d}"


def next_uc_id(fid):
    n = q1("SELECT COUNT(*) n FROM use_cases WHERE flagship_id=?", fid)["n"]
    return f"U{n+1:02d}{fid}"


def get_table_columns(table_name):
    conn = get_db()
    cursor = conn.execute(f"PRAGMA table_info({table_name})")
    cols = [r["name"] for r in cursor.fetchall()]
    conn.close()
    return cols


def sanitize_col_name(name):
    import re
    s = name.strip().lower()
    s = re.sub(r'\s+', '_', s)
    s = re.sub(r'[^a-z0-9_]', '', s)
    return s


# ── Helpers IA value ──────────────────────────────────────────────────────────
def get_value_entries(use_case_id=None, country=None, year=None):
    """Récupère les entrées de valeur, filtrables."""
    sql = "SELECT * FROM value_entries WHERE 1=1"
    params = []
    if use_case_id is not None:
        sql += " AND use_case_id=?"
        params.append(use_case_id)
    if country is not None:
        sql += " AND country=?"
        params.append(country)
    if year is not None:
        sql += " AND year=?"
        params.append(year)
    sql += " ORDER BY year, nature"
    return q(sql, *params)


def usecase_total_value(use_case_id):
    """Total IA value (toutes années, tous pays, toutes natures) d'un use case."""
    r = q1("SELECT COALESCE(SUM(amount),0) n FROM value_entries WHERE use_case_id=?", use_case_id)
    return r["n"] if r else 0


def usecase_value_by_nature(use_case_id):
    """Ventilation du total IA value d'un use case par nature."""
    rows = q("""
        SELECT nature, COALESCE(SUM(amount),0) total
        FROM value_entries WHERE use_case_id=?
        GROUP BY nature ORDER BY total DESC
    """, use_case_id)
    return {r["nature"]: r["total"] for r in rows}


In [ ]:
# === auth.py ===
import hashlib
from db import q1, run


def hash_pw(pw):
    return hashlib.sha256(pw.encode()).hexdigest()


def check_login(u, p):
    return bool(q1("SELECT 1 FROM users WHERE username=? AND password_hash=?", u, hash_pw(p)))


def is_user_admin(username):
    """Renvoie True si l'utilisateur a le flag admin en base."""
    r = q1("SELECT is_admin FROM users WHERE username=?", username)
    return bool(r and r["is_admin"])


def create_user(u, p, is_admin=False):
    try:
        run("INSERT INTO users (username, password_hash, is_admin) VALUES (?,?,?)",
            u, hash_pw(p), 1 if is_admin else 0)
        return True
    except Exception:
        return False


def is_flagship_authorized(username, flagship_id):
    if is_user_admin(username):
        return True
    return bool(q1("""
        SELECT 1 FROM user_countries uc_auth
        JOIN countries c ON uc_auth.country_id = c.id
        JOIN use_cases uc ON c.use_case_id = uc.id
        WHERE uc_auth.username = ? AND uc.flagship_id = ?
    """, username, flagship_id))


def is_usecase_authorized(username, usecase_id):
    if is_user_admin(username):
        return True
    return bool(q1("""
        SELECT 1 FROM user_countries uc_auth
        JOIN countries c ON uc_auth.country_id = c.id
        WHERE uc_auth.username = ? AND c.use_case_id = ?
    """, username, usecase_id))


def is_country_authorized(username, country_id):
    if is_user_admin(username):
        return True
    return bool(q1("SELECT 1 FROM user_countries WHERE username=? AND country_id=?", username, country_id))


def log_change(username, action, table_name, record_id, details):
    run("""
        INSERT INTO audit_logs (username, action, table_name, record_id, details)
        VALUES (?,?,?,?,?)
    """, username, action, table_name, str(record_id), details)


In [ ]:
# === ui_components.py ===
import streamlit as st
from db import VALUE_NATURES

COUNTRIES = ['France', 'Italy', 'Belgium', 'Luxembourg', 'Poland', 'Czech Republic', 'Romania',
             'Morocco', 'Algeria', 'Tunisia', 'Germany', 'Spain', 'UK', 'Switzerland', 'Netherlands', 'USA', 'Singapore', 'Japan']
AI_ACT    = ["Minimal Risk", "Limited Risk", "High Risk", "Unacceptable Risk"]
STATUS    = ["Draft", "Pilot", "Industrialized"]
MEASURE   = ["Estimated", "Measured"]
ROLLOUT   = ["Local", "Regional", "Global"]
GATES     = ["Gate 1", "Gate 2", "Gate 3", "Gate 4", "Gate 5"]
SOURCING  = ["In-house", "Vendor", "Hybrid"]
PLATFORMS = ["Azure ML", "GCP Vertex AI", "AWS SageMaker", "Databricks", "Internal LLM Gateway", "On-prem", "Azure", "AWS", "Other"]
MRR_CATS  = ["", "Operational Efficiency", "Risk Management", "Revenue", "Productivity", "Cost Reduction"]

PROTECTED_COLS = {
    "flagships": ["id", "name", "rank"],
    "use_cases": ["id", "flagship_id", "name"],
    "countries": ["id", "use_case_id", "country"]
}

SELECT_BOXES = {
    "ai_act": AI_ACT,
    "platform": PLATFORMS,
    "sourcing": SOURCING,
    "gate": GATES,
    "status": STATUS,
    "measure": MEASURE,
    "rollout": ROLLOUT,
    "mrr_cat": MRR_CATS
}

# Colonnes monétaires restantes (build au niveau pays)
MONEY_COLS = ("build",)


def fmt_money(v):
    try:
        return f"{int(round(float(v))):,} k€".replace(",", " ")
    except (TypeError, ValueError):
        return "—"


def render_field_input(table_name, col_name, value, key_prefix):
    if col_name in SELECT_BOXES:
        options = SELECT_BOXES[col_name]
        try:
            idx = options.index(value)
        except ValueError:
            idx = 0
        return st.selectbox(col_name.replace("_", " ").title(), options, index=idx, key=f"{key_prefix}_{col_name}")
    elif col_name == "description":
        return st.text_area("Description", value=value or "", key=f"{key_prefix}_{col_name}")
    elif col_name in MONEY_COLS:
        return st.number_input(col_name.replace("_", " ").title() + " (k€)", min_value=0, value=int(value or 0), key=f"{key_prefix}_{col_name}")
    elif col_name == "prod_date":
        return st.text_input("Date de production (YYYY-MM-DD)", value=str(value) if value else "", key=f"{key_prefix}_{col_name}")
    elif col_name == "rank":
        return st.number_input("Rang", min_value=1, max_value=99, value=int(value or 1), key=f"{key_prefix}_{col_name}")
    else:
        return st.text_input(col_name.replace("_", " ").title(), value=str(value) if value is not None else "", key=f"{key_prefix}_{col_name}")


def render_details_dynamic(row, table_name, exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = []
    cols = [k for k in row.keys() if k not in exclude_cols]
    if not cols:
        return
    col1, col2 = st.columns(2)
    for i, col_name in enumerate(cols):
        val = row[col_name]
        if val is None:
            display_val = "—"
        elif isinstance(val, (int, float)) and col_name in MONEY_COLS:
            display_val = fmt_money(val)
        else:
            display_val = str(val)
        label = col_name.replace("_", " ").title()
        with col1 if i % 2 == 0 else col2:
            st.write(f"**{label}** : {display_val}")


In [ ]:
# === quarters.py ===
"""
Logique de répartition de l'IA value par quarter.

Règle retenue : répartition ÉGALE de la valeur annuelle sur les quarters
RESTANTS de l'année, à partir du quarter de la date de mise en production.

Exemple : valeur annuelle = 1000, prod en 2026-Q3
          => Q1=0, Q2=0, Q3=500, Q4=500
"""
from datetime import datetime


def quarter_from_date(date_str):
    """Retourne le numéro de quarter (1-4) à partir d'une date 'YYYY-MM-DD'.
    Renvoie None si la date est absente ou invalide."""
    if not date_str:
        return None
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y"):
        try:
            d = datetime.strptime(str(date_str).strip(), fmt)
            return (d.month - 1) // 3 + 1
        except (ValueError, TypeError):
            continue
    return None


def year_from_date(date_str):
    """Retourne l'année (int) à partir d'une date, ou None."""
    if not date_str:
        return None
    for fmt in ("%Y-%m-%d", "%Y/%m/%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(str(date_str).strip(), fmt).year
        except (ValueError, TypeError):
            continue
    return None


def split_annual_to_quarters(annual_value, prod_date, target_year):
    """
    Répartit `annual_value` sur les 4 quarters de `target_year`.

    - Si l'année cible est l'année de prod : on répartit également sur les
      quarters restants (du quarter de prod jusqu'à Q4).
    - Si l'année cible est postérieure à l'année de prod : valeur pleine
      répartie sur les 4 quarters (le use case tourne toute l'année).
    - Si l'année cible est antérieure à l'année de prod : tout à 0.
    - Si pas de date de prod : on répartit également sur les 4 quarters.

    Retourne une liste [q1, q2, q3, q4].
    """
    annual_value = float(annual_value or 0)
    prod_year = year_from_date(prod_date)
    prod_q = quarter_from_date(prod_date)

    # Pas de date de prod connue -> on étale sur toute l'année cible
    if prod_year is None or prod_q is None:
        per_q = annual_value / 4
        return [per_q, per_q, per_q, per_q]

    if target_year < prod_year:
        return [0.0, 0.0, 0.0, 0.0]

    if target_year > prod_year:
        per_q = annual_value / 4
        return [per_q, per_q, per_q, per_q]

    # target_year == prod_year : quarters restants à partir du quarter de prod
    remaining = 4 - prod_q + 1  # nb de quarters restants (prod_q inclus)
    per_q = annual_value / remaining
    quarters = [0.0, 0.0, 0.0, 0.0]
    for qi in range(prod_q, 5):
        quarters[qi - 1] = per_q
    return quarters


def quarters_for_country_year(value_entries, prod_date, target_year):
    """
    Agrège une liste d'entrées de valeur (chacune {amount, nature, year})
    pour une année cible, et renvoie un dict :
      {
        "total_annual": float,
        "quarters": [q1,q2,q3,q4],
        "by_nature": { nature: annual_amount, ... },
        "by_nature_quarters": { nature: [q1,q2,q3,q4], ... }
      }
    """
    by_nature = {}
    for e in value_entries:
        if int(e.get("year") or 0) != int(target_year):
            continue
        nat = e.get("nature") or "Autre"
        by_nature[nat] = by_nature.get(nat, 0.0) + float(e.get("amount") or 0)

    by_nature_quarters = {}
    total_q = [0.0, 0.0, 0.0, 0.0]
    for nat, amt in by_nature.items():
        qs = split_annual_to_quarters(amt, prod_date, target_year)
        by_nature_quarters[nat] = qs
        for i in range(4):
            total_q[i] += qs[i]

    return {
        "total_annual": sum(by_nature.values()),
        "quarters": total_q,
        "by_nature": by_nature,
        "by_nature_quarters": by_nature_quarters,
    }


In [ ]:
# === pages.py ===
import streamlit as st
from db import (q, q1, run, next_f_id, next_uc_id, get_table_columns, sanitize_col_name,
                VALUE_NATURES, get_value_entries, usecase_total_value, usecase_value_by_nature)
from auth import (is_flagship_authorized, is_usecase_authorized, is_country_authorized,
                  log_change, check_login, create_user, is_user_admin)
from ui_components import (PROTECTED_COLS, COUNTRIES, render_field_input,
                           render_details_dynamic, fmt_money)
from quarters import quarters_for_country_year, split_annual_to_quarters


def _is_admin():
    return st.session_state.get("is_admin", False)


# ──────────────────────────────────────────────────────────────────────────────
def render_login_page():
    st.title("Flagship Registry")
    st.subheader("Connexion")
    u = st.text_input("Utilisateur", key="li_u")
    p = st.text_input("Mot de passe", type="password", key="li_p")
    if st.button("Se connecter", type="primary"):
        if check_login(u, p):
            st.session_state.logged_in = True
            st.session_state.username = u
            st.session_state.is_admin = is_user_admin(u)
            st.session_state.page = "list"
            st.rerun()
        else:
            st.error("Identifiants incorrects.")


# ── KPIs sensibles aux filtres ────────────────────────────────────────────────
def _filtered_value_total(filter_countries, filter_statuses, filter_divisions, username):
    """
    Calcule le total IA value en respectant les filtres actifs.
    Joint value_entries -> use_cases -> flagships, et countries (pour status).
    Pour un non-admin, restreint aux pays autorisés.
    """
    sql = """
        SELECT COALESCE(SUM(ve.amount),0) n
        FROM value_entries ve
        JOIN use_cases uc ON ve.use_case_id = uc.id
        JOIN flagships f ON uc.flagship_id = f.id
        JOIN countries c ON c.use_case_id = uc.id AND c.country = ve.country
    """
    where = []
    params = []
    if filter_countries:
        where.append("ve.country IN ({})".format(", ".join(["?"] * len(filter_countries))))
        params += filter_countries
    if filter_statuses:
        where.append("c.status IN ({})".format(", ".join(["?"] * len(filter_statuses))))
        params += filter_statuses
    if filter_divisions:
        where.append("f.division IN ({})".format(", ".join(["?"] * len(filter_divisions))))
        params += filter_divisions
    if not _is_admin():
        where.append("""c.id IN (SELECT country_id FROM user_countries WHERE username=?)""")
        params.append(username)
    if where:
        sql += " WHERE " + " AND ".join(where)
    r = q1(sql, *params)
    return r["n"] if r else 0


def _filtered_build_total(filter_countries, filter_statuses, filter_divisions, username):
    sql = """
        SELECT COALESCE(SUM(c.build),0) n
        FROM countries c
        JOIN use_cases uc ON c.use_case_id = uc.id
        JOIN flagships f ON uc.flagship_id = f.id
    """
    where = []
    params = []
    if filter_countries:
        where.append("c.country IN ({})".format(", ".join(["?"] * len(filter_countries))))
        params += filter_countries
    if filter_statuses:
        where.append("c.status IN ({})".format(", ".join(["?"] * len(filter_statuses))))
        params += filter_statuses
    if filter_divisions:
        where.append("f.division IN ({})".format(", ".join(["?"] * len(filter_divisions))))
        params += filter_divisions
    if not _is_admin():
        where.append("""c.id IN (SELECT country_id FROM user_countries WHERE username=?)""")
        params.append(username)
    if where:
        sql += " WHERE " + " AND ".join(where)
    r = q1(sql, *params)
    return r["n"] if r else 0


def render_list_page():
    st.title("Flagship Registry")

    fc = st.session_state.filter_countries
    fs = st.session_state.filter_statuses
    fd = st.session_state.filter_divisions
    uname = st.session_state.username
    filters_active = bool(fc or fs or fd)

    # ── KPIs (admin) qui suivent les filtres ──────────────────────────────────
    if _is_admin():
        # Comptages cohérents avec les filtres
        base = """
            FROM flagships f
            JOIN use_cases uc ON f.id = uc.flagship_id
            JOIN countries c ON uc.id = c.use_case_id
        """
        where, params = [], []
        if fc:
            where.append("c.country IN ({})".format(", ".join(["?"] * len(fc))))
            params += fc
        if fs:
            where.append("c.status IN ({})".format(", ".join(["?"] * len(fs))))
            params += fs
        if fd:
            where.append("f.division IN ({})".format(", ".join(["?"] * len(fd))))
            params += fd
        wsql = (" WHERE " + " AND ".join(where)) if where else ""

        f_count = q1(f"SELECT COUNT(DISTINCT f.id) n {base}{wsql}", *params)["n"] if filters_active else q1("SELECT COUNT(*) n FROM flagships")["n"]
        uc_count = q1(f"SELECT COUNT(DISTINCT uc.id) n {base}{wsql}", *params)["n"] if filters_active else q1("SELECT COUNT(*) n FROM use_cases")["n"]
        c_count = q1(f"SELECT COUNT(DISTINCT c.country) n {base}{wsql}", *params)["n"] if filters_active else q1("SELECT COUNT(DISTINCT country) n FROM countries")["n"]
        val = _filtered_value_total(fc, fs, fd, uname)
        build = _filtered_build_total(fc, fs, fd, uname)

        if filters_active:
            st.info("📊 Les KPIs ci-dessous reflètent les filtres actifs.")
        c1, c2, c3, c4, c5 = st.columns(5)
        c1.metric("Flagships", f_count)
        c2.metric("Use Cases", uc_count)
        c3.metric("IA Value", fmt_money(val))
        c4.metric("Build", fmt_money(build))
        c5.metric("Pays", c_count)
        st.divider()

    search = st.text_input("Rechercher", placeholder="Rechercher...")

    # ── Filtres avancés (admin) ───────────────────────────────────────────────
    if _is_admin():
        with st.expander("🔍 Filtres Avancés", expanded=filters_active):
            cf1, cf2, cf3 = st.columns(3)
            with cf1:
                all_countries = [r["country"] for r in q("SELECT DISTINCT country FROM countries ORDER BY country")]
                st.session_state.filter_countries = st.multiselect("Filtrer par Pays", all_countries, default=fc)
            with cf2:
                all_divisions = [r["division"] for r in q("SELECT DISTINCT division FROM flagships WHERE division IS NOT NULL AND division != '' ORDER BY division")]
                st.session_state.filter_divisions = st.multiselect("Filtrer par Division", all_divisions, default=fd)
            with cf3:
                all_statuses = [r["status"] for r in q("SELECT DISTINCT status FROM countries WHERE status IS NOT NULL AND status != '' ORDER BY status")]
                st.session_state.filter_statuses = st.multiselect("Filtrer par Statut", all_statuses, default=fs)
            if st.button("♻️ Réinitialiser les filtres"):
                st.session_state.filter_countries = []
                st.session_state.filter_divisions = []
                st.session_state.filter_statuses = []
                st.rerun()

        # Lien vers le dashboard détaillé
        if st.button("📈 Dashboard gains & coûts"):
            st.session_state.page = "dashboard"
            st.rerun()

    # rafraîchir les variables locales après mise à jour
    fc = st.session_state.filter_countries
    fs = st.session_state.filter_statuses
    fd = st.session_state.filter_divisions
    filters_active = bool(fc or fs or fd)

    # ── Liste des flagships autorisés ─────────────────────────────────────────
    if _is_admin():
        flagships = q("SELECT * FROM flagships ORDER BY rank")
        if fc:
            flagships = [
                f for f in flagships
                if q1("""
                    SELECT 1 FROM countries c
                    JOIN use_cases uc ON c.use_case_id = uc.id
                    WHERE uc.flagship_id = ? AND c.country IN ({})
                """.format(", ".join(["?"] * len(fc))), f["id"], *fc)
            ]
        if fd:
            flagships = [f for f in flagships if f.get("division") in fd]
        if fs:
            flagships = [
                f for f in flagships
                if q1("""
                    SELECT 1 FROM countries c
                    JOIN use_cases uc ON c.use_case_id = uc.id
                    WHERE uc.flagship_id = ? AND c.status IN ({})
                """.format(", ".join(["?"] * len(fs))), f["id"], *fs)
            ]
    else:
        flagships = q("""
            SELECT DISTINCT f.* FROM flagships f
            JOIN use_cases uc ON f.id = uc.flagship_id
            JOIN countries c ON uc.id = c.use_case_id
            JOIN user_countries uc_auth ON c.id = uc_auth.country_id
            WHERE uc_auth.username = ?
            ORDER BY f.rank
        """, st.session_state.username)

    if search:
        s = search.lower()
        flagships = [f for f in flagships if any(s in str(v).lower() for v in f.values() if v is not None)]

    if _is_admin():
        if st.button("＋ Nouveau flagship", type="primary"):
            st.session_state.page = "flagship_form"
            st.session_state.edit_flagship_id = None
            st.rerun()
        st.divider()

    for f in flagships:
        if _is_admin():
            if filters_active:
                sql = """
                    SELECT COUNT(DISTINCT uc.id) n FROM use_cases uc
                    JOIN countries c ON uc.id = c.use_case_id
                    WHERE uc.flagship_id = ?
                """
                params = [f["id"]]
                if fc:
                    sql += " AND c.country IN ({})".format(", ".join(["?"] * len(fc)))
                    params += fc
                if fs:
                    sql += " AND c.status IN ({})".format(", ".join(["?"] * len(fs)))
                    params += fs
                uc_n = q1(sql, *params)["n"]
            else:
                uc_n = q1("SELECT COUNT(*) n FROM use_cases WHERE flagship_id=?", f["id"])["n"]
        else:
            uc_n = q1("""
                SELECT COUNT(DISTINCT uc.id) n FROM use_cases uc
                JOIN countries c ON uc.id = c.use_case_id
                JOIN user_countries uc_auth ON c.id = uc_auth.country_id
                WHERE uc.flagship_id = ? AND uc_auth.username = ?
            """, f["id"], st.session_state.username)["n"]

        division_str = f"  ·  {f['division']}" if f.get("division") else ""
        with st.expander(f"#{f['rank']} — {f['name']}  ·  {uc_n} use case(s){division_str}"):
            if f.get("description"):
                st.write(f["description"])
            captions = [f"{k.capitalize()}: {f[k]}" for k in ("sponsor", "owner", "poc") if f.get(k)]
            if captions:
                st.caption("  |  ".join(captions))
            if st.button("Ouvrir", key=f"open_{f['id']}"):
                st.session_state.page = "flagship_detail"
                st.session_state.current_flagship_id = f["id"]
                st.rerun()


# ──────────────────────────────────────────────────────────────────────────────
def render_flagship_detail_page():
    fid = st.session_state.get("current_flagship_id")
    f = q1("SELECT * FROM flagships WHERE id=?", fid)
    if not f:
        st.error("Flagship introuvable.")
        st.stop()
    if not is_flagship_authorized(st.session_state.username, fid):
        st.error("Accès non autorisé.")
        st.stop()

    st.title(f["name"])
    st.caption(f"#{f['rank']}")
    if f.get("description"):
        st.write(f["description"])

    render_details_dynamic(f, "flagships", exclude_cols=["id", "name", "rank", "description"])
    st.write("")

    if _is_admin():
        col_edit, col_del = st.columns(2)
        with col_edit:
            if st.button("✎ Modifier le flagship"):
                st.session_state.page = "flagship_form"
                st.session_state.edit_flagship_id = fid
                st.rerun()
        with col_del:
            if st.button("🗑 Supprimer le flagship"):
                st.session_state["confirm_del_f"] = True
            if st.session_state.get("confirm_del_f"):
                # Compter ce qui va être détruit en cascade
                n_uc = q1("SELECT COUNT(*) n FROM use_cases WHERE flagship_id=?", fid)["n"]
                n_c = q1("""SELECT COUNT(*) n FROM countries c JOIN use_cases uc ON c.use_case_id=uc.id WHERE uc.flagship_id=?""", fid)["n"]
                n_v = q1("""SELECT COUNT(*) n FROM value_entries ve JOIN use_cases uc ON ve.use_case_id=uc.id WHERE uc.flagship_id=?""", fid)["n"]
                st.error(
                    f"⚠️ **SUPPRESSION DÉFINITIVE** ⚠️\n\n"
                    f"Vous êtes sur le point de supprimer le flagship **{f['name']}**.\n\n"
                    f"Cela effacera **DÉFINITIVEMENT de la base de données** :\n"
                    f"- **{n_uc}** use case(s)\n"
                    f"- **{n_c}** déploiement(s) pays\n"
                    f"- **{n_v}** entrée(s) de valeur IA (gains/coûts)\n\n"
                    f"**Cette action est IRRÉVERSIBLE.**"
                )
                confirm = st.text_input("Tapez SUPPRIMER pour confirmer", key="confirm_txt_f")
                cc1, cc2 = st.columns(2)
                with cc1:
                    if st.button("Oui, tout supprimer", type="primary", disabled=(confirm != "SUPPRIMER")):
                        ucs = [r["id"] for r in q("SELECT id FROM use_cases WHERE flagship_id=?", fid)]
                        for ucid in ucs:
                            run("DELETE FROM value_entries WHERE use_case_id=?", ucid)
                            run("DELETE FROM countries WHERE use_case_id=?", ucid)
                        run("DELETE FROM use_cases WHERE flagship_id=?", fid)
                        log_change(st.session_state.username, "DELETE", "flagships", fid,
                                   f"Suppression flagship {f['name']} (+{n_uc} UC, {n_c} pays, {n_v} valeurs)")
                        run("DELETE FROM flagships WHERE id=?", fid)
                        st.session_state.page = "list"
                        st.session_state.pop("confirm_del_f", None)
                        st.rerun()
                with cc2:
                    if st.button("Annuler"):
                        st.session_state.pop("confirm_del_f", None)
                        st.rerun()
    else:
        if st.button("✎ Modifier le flagship"):
            st.session_state.page = "flagship_form"
            st.session_state.edit_flagship_id = fid
            st.rerun()

    st.divider()
    st.subheader("Use Cases")

    if _is_admin():
        if st.button("＋ Ajouter un use case", type="primary"):
            st.session_state.page = "usecase_form"
            st.session_state.edit_usecase_id = None
            st.rerun()

    fc = st.session_state.filter_countries
    fs = st.session_state.filter_statuses

    if _is_admin():
        if fc or fs:
            sql = "SELECT DISTINCT uc.* FROM use_cases uc JOIN countries c ON uc.id = c.use_case_id WHERE uc.flagship_id = ?"
            params = [fid]
            if fc:
                sql += " AND c.country IN ({})".format(", ".join(["?"] * len(fc)))
                params += fc
            if fs:
                sql += " AND c.status IN ({})".format(", ".join(["?"] * len(fs)))
                params += fs
            sql += " ORDER BY uc.id"
            use_cases = q(sql, *params)
        else:
            use_cases = q("SELECT * FROM use_cases WHERE flagship_id=? ORDER BY id", fid)
    else:
        use_cases = q("""
            SELECT DISTINCT uc.* FROM use_cases uc
            JOIN countries c ON uc.id = c.use_case_id
            JOIN user_countries uc_auth ON c.id = uc_auth.country_id
            WHERE uc.flagship_id = ? AND uc_auth.username = ?
            ORDER BY uc.id
        """, fid, st.session_state.username)

    if not use_cases:
        st.info("Aucun use case accessible pour ce flagship.")
    for uc in use_cases:
        if _is_admin():
            if fc or fs:
                sql = "SELECT * FROM countries WHERE use_case_id = ?"
                params = [uc["id"]]
                if fc:
                    sql += " AND country IN ({})".format(", ".join(["?"] * len(fc)))
                    params += fc
                if fs:
                    sql += " AND status IN ({})".format(", ".join(["?"] * len(fs)))
                    params += fs
                sql += " ORDER BY country"
                countries = q(sql, *params)
            else:
                countries = q("SELECT * FROM countries WHERE use_case_id=? ORDER BY country", uc["id"])
        else:
            countries = q("""
                SELECT c.* FROM countries c
                JOIN user_countries uc_auth ON c.id = uc_auth.country_id
                WHERE c.use_case_id = ? AND uc_auth.username = ?
                ORDER BY c.country
            """, uc["id"], st.session_state.username)

        ai_act_val = uc.get("ai_act", "")
        ai_act_str = f"  ·  {ai_act_val}" if ai_act_val else ""
        uc_val = usecase_total_value(uc["id"])
        with st.expander(f"{uc['name']}{ai_act_str}  ·  {len(countries)} pays  ·  {fmt_money(uc_val)}"):
            if uc.get("description"):
                st.write(uc["description"])
            cap = []
            if uc.get("technology"):
                cap.append(f"Tech: {uc['technology']}")
            if uc.get("platform"):
                cap.append(f"Platform: {uc['platform']}")
            if uc.get("gate"):
                cap.append(f"Gate: {uc['gate']}")
            if cap:
                st.caption("  |  ".join(cap))

            ca, cb = st.columns(2)
            with ca:
                if st.button("Ouvrir", key=f"o_{uc['id']}"):
                    st.session_state.page = "usecase_detail"
                    st.session_state.current_usecase_id = uc["id"]
                    st.rerun()
            with cb:
                if st.button("✎ Modifier", key=f"e_{uc['id']}"):
                    st.session_state.page = "usecase_form"
                    st.session_state.edit_usecase_id = uc["id"]
                    st.rerun()


# ──────────────────────────────────────────────────────────────────────────────
def render_flagship_form_page():
    fid = st.session_state.get("edit_flagship_id")
    f = q1("SELECT * FROM flagships WHERE id=?", fid) if fid else {}
    is_new = not bool(f)

    if fid and not is_flagship_authorized(st.session_state.username, fid):
        st.error("Accès non autorisé.")
        st.stop()

    st.title("Nouveau flagship" if is_new else "Modifier le flagship")
    cols = get_table_columns("flagships")
    extra_cols = [c for c in cols if c not in ("id", "name", "rank")]

    form_values = {}
    with st.form("f_form"):
        name = st.text_input("Nom *", value=f.get("name", ""))
        rank = st.number_input("Rang", min_value=1, max_value=99, value=int(f.get("rank", 1)))
        if extra_cols:
            col1, col2 = st.columns(2)
            for i, col_name in enumerate(extra_cols):
                val = f.get(col_name, "")
                with col1 if i % 2 == 0 else col2:
                    form_values[col_name] = render_field_input("flagships", col_name, val, "fs")
        if st.form_submit_button("💾 Sauvegarder", type="primary"):
            if not name:
                st.error("Le nom est obligatoire.")
            else:
                new_id = fid or next_f_id()
                data = {"id": new_id, "name": name, "rank": rank}
                for col_name in extra_cols:
                    data[col_name] = form_values[col_name]
                if is_new:
                    keys = list(data.keys())
                    placeholders = ", ".join(["?"] * len(keys))
                    run(f"INSERT INTO flagships ({', '.join(keys)}) VALUES ({placeholders})", *list(data.values()))
                    log_change(st.session_state.username, "INSERT", "flagships", new_id, f"Création du flagship: {name}")
                else:
                    keys = [k for k in data.keys() if k != "id"]
                    assignments = ", ".join([f"{k}=?" for k in keys])
                    values = [data[k] for k in keys] + [new_id]
                    run(f"UPDATE flagships SET {assignments} WHERE id=?", *values)
                    log_change(st.session_state.username, "UPDATE", "flagships", new_id, f"Modification du flagship: {name} (Rang: {rank})")
                st.session_state.current_flagship_id = new_id
                st.session_state.page = "flagship_detail"
                st.rerun()

    if _is_admin():
        _render_column_config("flagships", cols)


# ── Helper factorisé : configuration dynamique des colonnes (admin) ───────────
def _render_column_config(table_name, cols):
    label = {"flagships": "Flagships", "use_cases": "Use Cases", "countries": "Countries"}[table_name]
    placeholder = {
        "flagships": "ex: priorite_division, budget...",
        "use_cases": "ex: budget, ref_tech...",
        "countries": "ex: priorite, contact_local...",
    }[table_name]
    st.write("---")
    st.subheader(f"⚙️ Configurer les colonnes de la table {label}")
    st.warning("⚠️ ATTENTION : Supprimer une colonne supprimera définitivement les données correspondantes.")

    ca1, ca2 = st.columns([3, 1])
    with ca1:
        new_field = st.text_input("Nom du nouveau champ", placeholder=placeholder, key=f"new_field_{table_name}")
    with ca2:
        st.write(" ")
        st.write(" ")
        if st.button("➕ Ajouter", key=f"btn_add_{table_name}"):
            if new_field:
                sanitized = sanitize_col_name(new_field)
                existing = get_table_columns(table_name)
                if not sanitized:
                    st.error("Nom de champ invalide.")
                elif sanitized in existing:
                    st.error("Ce champ existe déjà.")
                else:
                    run(f"ALTER TABLE {table_name} ADD COLUMN {sanitized} TEXT")
                    log_change(st.session_state.username, "SCHEMA_ADD", table_name, sanitized, f"Ajout de la colonne: {sanitized}")
                    st.success(f"Champ '{sanitized}' ajouté !")
                    st.rerun()
            else:
                st.error("Veuillez saisir un nom.")

    st.write("**Champs personnalisables existants :**")
    deletable_cols = [c for c in cols if c not in PROTECTED_COLS[table_name]]
    if deletable_cols:
        for c in deletable_cols:
            cl1, cl2 = st.columns([3, 1])
            with cl1:
                st.code(c)
            with cl2:
                if st.button("🗑️ Supprimer", key=f"del_{table_name}_{c}"):
                    run(f"ALTER TABLE {table_name} DROP COLUMN {c}")
                    log_change(st.session_state.username, "SCHEMA_DROP", table_name, c, f"Suppression de la colonne: {c}")
                    st.success(f"Champ '{c}' supprimé !")
                    st.rerun()
    else:
        st.info("Aucun champ personnalisable.")


# ──────────────────────────────────────────────────────────────────────────────
def render_usecase_detail_page():
    ucid = st.session_state.get("current_usecase_id")
    uc = q1("SELECT * FROM use_cases WHERE id=?", ucid)
    if not uc:
        st.error("Use case introuvable.")
        st.stop()
    if not is_usecase_authorized(st.session_state.username, ucid):
        st.error("Accès non autorisé.")
        st.stop()

    st.title(uc["name"])
    st.caption(f"{uc['id']}")
    if uc.get("description"):
        st.write(uc["description"])

    render_details_dynamic(uc, "use_cases", exclude_cols=["id", "flagship_id", "name", "description"])
    st.write("")

    # ── IA Value consolidée du use case + détail par nature ───────────────────
    st.subheader("💰 IA Value (consolidée use case)")
    by_nature = usecase_value_by_nature(ucid)
    total = sum(by_nature.values())
    if total == 0:
        st.info("Aucune valeur IA renseignée pour ce use case.")
    else:
        st.metric("Total IA Value", fmt_money(total))
        st.write("**Détail par nature :**")
        cols_n = st.columns(min(len(by_nature), 5) or 1)
        for i, (nat, amt) in enumerate(by_nature.items()):
            cols_n[i % len(cols_n)].metric(nat, fmt_money(amt))

    st.write("")
    if _is_admin():
        ce, cd = st.columns(2)
        with ce:
            if st.button("✎ Modifier le use case"):
                st.session_state.page = "usecase_form"
                st.session_state.edit_usecase_id = ucid
                st.rerun()
        with cd:
            if st.button("🗑 Supprimer le use case"):
                st.session_state["confirm_del_uc"] = True
            if st.session_state.get("confirm_del_uc"):
                n_c = q1("SELECT COUNT(*) n FROM countries WHERE use_case_id=?", ucid)["n"]
                n_v = q1("SELECT COUNT(*) n FROM value_entries WHERE use_case_id=?", ucid)["n"]
                st.error(
                    f"⚠️ **SUPPRESSION DÉFINITIVE** ⚠️\n\n"
                    f"Supprimer le use case **{uc['name']}** effacera **DÉFINITIVEMENT** :\n"
                    f"- **{n_c}** déploiement(s) pays\n"
                    f"- **{n_v}** entrée(s) de valeur IA\n\n"
                    f"**Action IRRÉVERSIBLE.**"
                )
                cc1, cc2 = st.columns(2)
                with cc1:
                    if st.button("Oui, supprimer", type="primary"):
                        run("DELETE FROM value_entries WHERE use_case_id=?", ucid)
                        run("DELETE FROM countries WHERE use_case_id=?", ucid)
                        log_change(st.session_state.username, "DELETE", "use_cases", ucid,
                                   f"Suppression use case {uc['name']} (+{n_c} pays, {n_v} valeurs)")
                        run("DELETE FROM use_cases WHERE id=?", ucid)
                        st.session_state.page = "flagship_detail"
                        st.session_state.pop("confirm_del_uc", None)
                        st.rerun()
                with cc2:
                    if st.button("Annuler"):
                        st.session_state.pop("confirm_del_uc", None)
                        st.rerun()
    else:
        if st.button("✎ Modifier le use case"):
            st.session_state.page = "usecase_form"
            st.session_state.edit_usecase_id = ucid
            st.rerun()

    st.divider()
    st.subheader("Déploiements pays")

    if _is_admin():
        if st.button("＋ Ajouter un pays", type="primary"):
            st.session_state.page = "country_form"
            st.session_state.edit_country = None
            st.rerun()

    fc = st.session_state.filter_countries
    fs = st.session_state.filter_statuses
    if _is_admin():
        if fc or fs:
            sql = "SELECT * FROM countries WHERE use_case_id = ?"
            params = [ucid]
            if fc:
                sql += " AND country IN ({})".format(", ".join(["?"] * len(fc)))
                params += fc
            if fs:
                sql += " AND status IN ({})".format(", ".join(["?"] * len(fs)))
                params += fs
            sql += " ORDER BY country"
            countries = q(sql, *params)
        else:
            countries = q("SELECT * FROM countries WHERE use_case_id=? ORDER BY country", ucid)
    else:
        countries = q("""
            SELECT c.* FROM countries c
            JOIN user_countries uc_auth ON c.id = uc_auth.country_id
            WHERE c.use_case_id = ? AND uc_auth.username = ?
            ORDER BY c.country
        """, ucid, st.session_state.username)

    if not countries:
        st.info("Aucun pays configuré ou accessible.")
    for c in countries:
        status_val = c.get("status", "")
        status_str = f"  ·  {status_val}" if status_val else ""
        # total valeur pays (toutes années/natures)
        cval = q1("SELECT COALESCE(SUM(amount),0) n FROM value_entries WHERE use_case_id=? AND country=?", ucid, c["country"])["n"]
        cval_str = f"  ·  {fmt_money(cval)}" if cval else ""
        with st.expander(f"{c['country']}{status_str}{cval_str}"):
            render_details_dynamic(c, "countries", exclude_cols=["id", "use_case_id", "country"])

            # Valeurs par année / nature pour ce pays
            entries = get_value_entries(use_case_id=ucid, country=c["country"])
            if entries:
                st.write("**IA Value par année et nature :**")
                years = sorted({e["year"] for e in entries})
                for yr in years:
                    yr_entries = [e for e in entries if e["year"] == yr]
                    parts = [f"{e['nature']}: {fmt_money(e['amount'])}" for e in yr_entries]
                    st.caption(f"**{yr}** — " + "  |  ".join(parts))

            st.write("")
            if _is_admin():
                cma, cda = st.columns(2)
                with cma:
                    if st.button("✎ Modifier", key=f"mc_{c['id']}"):
                        st.session_state.page = "country_form"
                        st.session_state.edit_country = c["country"]
                        st.rerun()
                with cda:
                    if st.button("🗑 Supprimer", key=f"dc_{c['id']}"):
                        st.session_state[f"confirm_del_c_{c['id']}"] = True
                    if st.session_state.get(f"confirm_del_c_{c['id']}"):
                        n_v = q1("SELECT COUNT(*) n FROM value_entries WHERE use_case_id=? AND country=?", ucid, c["country"])["n"]
                        st.error(f"⚠️ Supprimer **{c['country']}** effacera aussi **{n_v}** entrée(s) de valeur IA. Action irréversible.")
                        if st.button("Confirmer la suppression", type="primary", key=f"cfm_dc_{c['id']}"):
                            run("DELETE FROM value_entries WHERE use_case_id=? AND country=?", ucid, c["country"])
                            log_change(st.session_state.username, "DELETE", "countries", f"{ucid}/{c['country']}",
                                       f"Suppression déploiement pays {c['country']} (+{n_v} valeurs)")
                            run("DELETE FROM countries WHERE use_case_id=? AND country=?", ucid, c["country"])
                            st.session_state.pop(f"confirm_del_c_{c['id']}", None)
                            st.rerun()
            else:
                if st.button("✎ Modifier", key=f"mc_{c['id']}"):
                    st.session_state.page = "country_form"
                    st.session_state.edit_country = c["country"]
                    st.rerun()


# ──────────────────────────────────────────────────────────────────────────────
def render_usecase_form_page():
    ucid = st.session_state.get("edit_usecase_id")
    uc = q1("SELECT * FROM use_cases WHERE id=?", ucid) if ucid else {}
    fid = uc.get("flagship_id") if uc else st.session_state.get("current_flagship_id")
    is_new = not bool(uc)

    if ucid and not is_usecase_authorized(st.session_state.username, ucid):
        st.error("Accès non autorisé.")
        st.stop()

    st.title("Nouveau use case" if is_new else "Modifier le use case")
    cols = get_table_columns("use_cases")
    extra_cols = [c for c in cols if c not in ("id", "flagship_id", "name")]

    form_values = {}
    with st.form("uc_form"):
        name = st.text_input("Nom *", value=uc.get("name", ""))
        if extra_cols:
            col1, col2 = st.columns(2)
            for i, col_name in enumerate(extra_cols):
                val = uc.get(col_name, "")
                with col1 if i % 2 == 0 else col2:
                    form_values[col_name] = render_field_input("use_cases", col_name, val, "uc")
        if st.form_submit_button("💾 Sauvegarder", type="primary"):
            if not name:
                st.error("Le nom est obligatoire.")
            else:
                new_id = ucid or next_uc_id(fid)
                data = {"id": new_id, "flagship_id": fid, "name": name}
                for col_name in extra_cols:
                    data[col_name] = form_values[col_name]
                if is_new:
                    keys = list(data.keys())
                    placeholders = ", ".join(["?"] * len(keys))
                    run(f"INSERT INTO use_cases ({', '.join(keys)}) VALUES ({placeholders})", *list(data.values()))
                    log_change(st.session_state.username, "INSERT", "use_cases", new_id, f"Création du use case: {name}")
                else:
                    keys = [k for k in data.keys() if k != "id"]
                    assignments = ", ".join([f"{k}=?" for k in keys])
                    values = [data[k] for k in keys] + [new_id]
                    run(f"UPDATE use_cases SET {assignments} WHERE id=?", *values)
                    log_change(st.session_state.username, "UPDATE", "use_cases", new_id, f"Modification du use case: {name}")
                st.session_state.current_usecase_id = new_id
                st.session_state.page = "flagship_detail"
                st.rerun()

    if _is_admin():
        _render_column_config("use_cases", cols)


# ──────────────────────────────────────────────────────────────────────────────
def render_country_form_page():
    ucid   = st.session_state.get("current_usecase_id")
    edit_c = st.session_state.get("edit_country")
    c_data = q1("SELECT * FROM countries WHERE use_case_id=? AND country=?", ucid, edit_c) if edit_c else {}
    is_new = not bool(c_data)
    used = [r["country"] for r in q("SELECT country FROM countries WHERE use_case_id=?", ucid)]
    available = [c for c in COUNTRIES if c not in used or c == edit_c]

    if edit_c and not _is_admin():
        if c_data and not is_country_authorized(st.session_state.username, c_data["id"]):
            st.error("Accès non autorisé.")
            st.stop()

    st.title("Ajouter un pays" if is_new else f"Modifier : {edit_c}")
    cols = get_table_columns("countries")
    # extra cols hors build (build a son propre champ) et prod_date géré normalement
    extra_cols = [c for c in cols if c not in ("id", "use_case_id", "country")]

    form_values = {}
    with st.form("c_form"):
        st.write("**Identification & déploiement**")
        country = st.selectbox("Pays *", available, index=available.index(edit_c) if edit_c in available else 0, disabled=not is_new)
        if extra_cols:
            col1, col2 = st.columns(2)
            for i, col_name in enumerate(extra_cols):
                val = c_data.get(col_name, "")
                with col1 if i % 2 == 0 else col2:
                    form_values[col_name] = render_field_input("countries", col_name, val, "co")
        if st.form_submit_button("💾 Sauvegarder", type="primary"):
            data = {"use_case_id": ucid, "country": country}
            for col_name in extra_cols:
                data[col_name] = form_values[col_name]
            if is_new:
                keys = list(data.keys())
                placeholders = ", ".join(["?"] * len(keys))
                run(f"INSERT INTO countries ({', '.join(keys)}) VALUES ({placeholders})", *list(data.values()))
                log_change(st.session_state.username, "INSERT", "countries", f"{ucid}/{country}", f"Création du déploiement pays: {country}")
            else:
                keys = [k for k in data.keys() if k not in ("use_case_id", "country")]
                assignments = ", ".join([f"{k}=?" for k in keys])
                values = [data[k] for k in keys] + [ucid, edit_c]
                run(f"UPDATE countries SET {assignments} WHERE use_case_id=? AND country=?", *values)
                log_change(st.session_state.username, "UPDATE", "countries", f"{ucid}/{edit_c}", f"Modification du déploiement pays: {edit_c}")
            st.session_state.edit_country = country
            st.rerun()

    # ── Éditeur IA Value par année et par nature (hors form principal) ────────
    if not is_new or edit_c:
        target_country = edit_c or st.session_state.get("edit_country")
        if target_country and q1("SELECT 1 FROM countries WHERE use_case_id=? AND country=?", ucid, target_country):
            st.divider()
            st.subheader("💰 IA Value par année et par nature")
            prod_date = q1("SELECT prod_date FROM countries WHERE use_case_id=? AND country=?", ucid, target_country)["prod_date"]
            st.caption(f"Date de production : {prod_date or '—'}  ·  La répartition par quarter en découle automatiquement.")

            existing = get_value_entries(use_case_id=ucid, country=target_country)
            if existing:
                st.write("**Entrées existantes :**")
                for e in existing:
                    qs = split_annual_to_quarters(e["amount"], prod_date, e["year"])
                    cqa, cqb = st.columns([4, 1])
                    with cqa:
                        st.write(f"**{e['year']}** · {e['nature']} : {fmt_money(e['amount'])}  "
                                 f"→ Q1 {fmt_money(qs[0])} · Q2 {fmt_money(qs[1])} · Q3 {fmt_money(qs[2])} · Q4 {fmt_money(qs[3])}")
                    with cqb:
                        if st.button("🗑", key=f"delval_{e['id']}"):
                            run("DELETE FROM value_entries WHERE id=?", e["id"])
                            log_change(st.session_state.username, "DELETE", "value_entries", e["id"],
                                       f"Suppression valeur {e['year']}/{e['nature']} ({target_country})")
                            st.rerun()

            st.write("**Ajouter / mettre à jour une entrée :**")
            with st.form("val_form"):
                vc1, vc2, vc3 = st.columns(3)
                with vc1:
                    yr = st.number_input("Année", min_value=2020, max_value=2035, value=2026)
                with vc2:
                    nat = st.selectbox("Nature", VALUE_NATURES)
                with vc3:
                    amt = st.number_input("Montant annuel (k€)", min_value=0, value=0)
                if st.form_submit_button("💾 Enregistrer la valeur", type="primary"):
                    run("""INSERT INTO value_entries (use_case_id, country, year, nature, amount)
                           VALUES (?,?,?,?,?)
                           ON CONFLICT(use_case_id, country, year, nature)
                           DO UPDATE SET amount=excluded.amount""",
                        ucid, target_country, int(yr), nat, float(amt))
                    log_change(st.session_state.username, "UPSERT", "value_entries", f"{ucid}/{target_country}/{yr}/{nat}",
                               f"Valeur {nat} {yr} = {amt} k€ ({target_country})")
                    qs = split_annual_to_quarters(amt, prod_date, int(yr))
                    st.success(f"Enregistré. Répartition {yr} → Q1 {fmt_money(qs[0])} / Q2 {fmt_money(qs[1])} / Q3 {fmt_money(qs[2])} / Q4 {fmt_money(qs[3])}")
                    st.rerun()

    if _is_admin():
        _render_column_config("countries", cols)


# ──────────────────────────────────────────────────────────────────────────────
def render_dashboard_page():
    if not _is_admin():
        st.error("Accès réservé à l'administrateur.")
        st.stop()

    st.title("📈 Dashboard gains & coûts")
    st.caption("Vue annuelle et trimestrielle, basée sur la date de production de chaque déploiement.")

    # ── Filtres dashboard ─────────────────────────────────────────────────────
    all_years = [r["year"] for r in q("SELECT DISTINCT year FROM value_entries ORDER BY year")]
    if not all_years:
        st.info("Aucune valeur IA en base.")
        st.stop()

    fc1, fc2, fc3, fc4 = st.columns(4)
    with fc1:
        sel_year = st.selectbox("Année", all_years, index=len(all_years) - 1)
    with fc2:
        all_div = [r["division"] for r in q("SELECT DISTINCT division FROM flagships WHERE division IS NOT NULL AND division!='' ORDER BY division")]
        f_div = st.multiselect("Division", all_div)
    with fc3:
        all_ctry = [r["country"] for r in q("SELECT DISTINCT country FROM countries ORDER BY country")]
        f_ctry = st.multiselect("Pays", all_ctry)
    with fc4:
        all_nat = VALUE_NATURES
        f_nat = st.multiselect("Nature", all_nat)

    # ── Récupération des valeurs de l'année + jointures pour filtres ──────────
    sql = """
        SELECT ve.use_case_id, ve.country, ve.year, ve.nature, ve.amount,
               c.prod_date, c.status, c.build, f.division, uc.flagship_id, uc.name as uc_name
        FROM value_entries ve
        JOIN use_cases uc ON ve.use_case_id = uc.id
        JOIN flagships f ON uc.flagship_id = f.id
        JOIN countries c ON c.use_case_id = uc.id AND c.country = ve.country
        WHERE ve.year = ?
    """
    params = [sel_year]
    if f_div:
        sql += " AND f.division IN ({})".format(", ".join(["?"] * len(f_div)))
        params += f_div
    if f_ctry:
        sql += " AND ve.country IN ({})".format(", ".join(["?"] * len(f_ctry)))
        params += f_ctry
    if f_nat:
        sql += " AND ve.nature IN ({})".format(", ".join(["?"] * len(f_nat)))
        params += f_nat
    rows = q(sql, *params)

    # ── Agrégats : total annuel + répartition par quarter ─────────────────────
    total_annual = 0.0
    quarters = [0.0, 0.0, 0.0, 0.0]
    by_nature = {}
    by_nature_q = {n: [0.0, 0.0, 0.0, 0.0] for n in VALUE_NATURES}
    for r in rows:
        total_annual += float(r["amount"])
        by_nature[r["nature"]] = by_nature.get(r["nature"], 0.0) + float(r["amount"])
        qs = split_annual_to_quarters(r["amount"], r["prod_date"], sel_year)
        for i in range(4):
            quarters[i] += qs[i]
            by_nature_q.setdefault(r["nature"], [0, 0, 0, 0])[i] += qs[i]

    # Build (coûts) : déploiements concernés par les filtres (hors nature)
    bsql = """
        SELECT COALESCE(SUM(c.build),0) n
        FROM countries c
        JOIN use_cases uc ON c.use_case_id = uc.id
        JOIN flagships f ON uc.flagship_id = f.id
        WHERE 1=1
    """
    bparams = []
    if f_div:
        bsql += " AND f.division IN ({})".format(", ".join(["?"] * len(f_div)))
        bparams += f_div
    if f_ctry:
        bsql += " AND c.country IN ({})".format(", ".join(["?"] * len(f_ctry)))
        bparams += f_ctry
    total_build = q1(bsql, *bparams)["n"]

    st.divider()
    m1, m2, m3, m4 = st.columns(4)
    m1.metric(f"Gains {sel_year}", fmt_money(total_annual))
    m2.metric("Build (coûts)", fmt_money(total_build))
    m3.metric("Net", fmt_money(total_annual - total_build))
    m4.metric("Déploiements", len({(r['use_case_id'], r['country']) for r in rows}))

    # ── Répartition par quarter ───────────────────────────────────────────────
    st.subheader(f"Gains par quarter — {sel_year}")
    qc1, qc2, qc3, qc4 = st.columns(4)
    qc1.metric("Q1", fmt_money(quarters[0]))
    qc2.metric("Q2", fmt_money(quarters[1]))
    qc3.metric("Q3", fmt_money(quarters[2]))
    qc4.metric("Q4", fmt_money(quarters[3]))

    try:
        import pandas as pd
        df_q = pd.DataFrame({"Quarter": ["Q1", "Q2", "Q3", "Q4"], "Gains (k€)": quarters})
        st.bar_chart(df_q.set_index("Quarter"))
    except Exception:
        pass

    # ── Détail par nature ─────────────────────────────────────────────────────
    st.subheader("Détail par nature")
    if by_nature:
        try:
            import pandas as pd
            data = []
            for nat, amt in by_nature.items():
                qn = by_nature_q.get(nat, [0, 0, 0, 0])
                data.append({"Nature": nat, "Annuel": amt, "Q1": qn[0], "Q2": qn[1], "Q3": qn[2], "Q4": qn[3]})
            df = pd.DataFrame(data).sort_values("Annuel", ascending=False)
            st.dataframe(df.style.format({c: "{:,.0f}".format for c in ["Annuel", "Q1", "Q2", "Q3", "Q4"]}), use_container_width=True)
        except Exception:
            for nat, amt in sorted(by_nature.items(), key=lambda x: -x[1]):
                st.write(f"**{nat}** : {fmt_money(amt)}")
    else:
        st.info("Aucune donnée pour ces filtres.")


# ──────────────────────────────────────────────────────────────────────────────
def render_users_admin_page():
    if not _is_admin():
        st.error("Accès réservé à l'administrateur.")
        st.stop()

    st.title("⚙️ Paramètres Habilitations")
    st.subheader("👥 Gestion des utilisateurs et des droits")

    deployments = q("""
        SELECT c.id as country_id, f.name as flagship_name, uc.name as usecase_name, c.country as country_name
        FROM countries c
        JOIN use_cases uc ON c.use_case_id = uc.id
        JOIN flagships f ON uc.flagship_id = f.id
        ORDER BY f.rank, uc.id, c.country
    """)
    dep_options = {d["country_id"]: f"{d['flagship_name']} ➔ {d['usecase_name']} ➔ {d['country_name']}" for d in deployments}

    st.write("### ➕ Créer un nouvel utilisateur")
    with st.form("create_user_form"):
        new_u = st.text_input("Nom d'utilisateur", key="adm_new_u")
        new_p = st.text_input("Mot de passe", type="password", key="adm_new_p")
        make_admin = st.checkbox("👑 Cet utilisateur est administrateur", key="adm_new_is_admin")
        st.caption("Un administrateur a accès à tout (KPIs, dashboard, configuration) sans assignation de pays.")
        selected_deps = st.multiselect(
            "Assigner des déploiements (pays / use case / flagship)",
            list(dep_options.keys()),
            format_func=lambda x: dep_options[x],
            disabled=make_admin
        )
        if st.form_submit_button("Créer l'utilisateur", type="primary"):
            if not new_u or not new_p:
                st.error("Veuillez remplir le nom d'utilisateur et le mot de passe.")
            elif new_u == "admin":
                st.error("Le nom d'utilisateur 'admin' est réservé.")
            elif create_user(new_u, new_p, is_admin=make_admin):
                if not make_admin:
                    for cid in selected_deps:
                        run("INSERT INTO user_countries (username, country_id) VALUES (?,?)", new_u, cid)
                role = "administrateur" if make_admin else "utilisateur standard"
                log_change(st.session_state.username, "INSERT", "users", new_u, f"Création {role}: {new_u}")
                st.success(f"Utilisateur '{new_u}' créé ({role}) !")
                st.rerun()
            else:
                st.error("Ce nom d'utilisateur existe déjà.")

    st.divider()
    st.write("### 👥 Utilisateurs existants")
    users = q("SELECT username, is_admin FROM users WHERE username != 'admin' ORDER BY username")
    if not users:
        st.info("Aucun autre utilisateur enregistré.")
    else:
        for u in users:
            uname = u["username"]
            badge = "👑 Admin" if u["is_admin"] else "👤 Standard"
            with st.expander(f"{badge} — {uname}"):
                is_adm = st.checkbox("👑 Administrateur", value=bool(u["is_admin"]), key=f"adm_flag_{uname}")
                assigned = [r["country_id"] for r in q("SELECT country_id FROM user_countries WHERE username=?", uname)]
                new_selected = st.multiselect(
                    f"Habilitations pour {uname}",
                    list(dep_options.keys()),
                    default=assigned,
                    format_func=lambda x: dep_options[x],
                    key=f"sel_deps_{uname}",
                    disabled=is_adm
                )
                cs, cd = st.columns(2)
                with cs:
                    if st.button("💾 Enregistrer", key=f"save_user_{uname}"):
                        run("UPDATE users SET is_admin=? WHERE username=?", 1 if is_adm else 0, uname)
                        run("DELETE FROM user_countries WHERE username=?", uname)
                        if not is_adm:
                            for cid in new_selected:
                                run("INSERT INTO user_countries (username, country_id) VALUES (?,?)", uname, cid)
                        log_change(st.session_state.username, "UPDATE", "users", uname,
                                   f"MAJ habilitations {uname} (admin={is_adm})")
                        st.success("Modifications enregistrées !")
                        st.rerun()
                with cd:
                    if st.button("🗑️ Supprimer", key=f"del_user_{uname}", type="primary"):
                        run("DELETE FROM users WHERE username=?", uname)
                        run("DELETE FROM user_countries WHERE username=?", uname)
                        log_change(st.session_state.username, "DELETE", "users", uname, f"Suppression utilisateur: {uname}")
                        st.success("Utilisateur supprimé !")
                        st.rerun()


# ──────────────────────────────────────────────────────────────────────────────
def render_audit_history_page():
    if not _is_admin():
        st.error("Accès réservé à l'administrateur.")
        st.stop()

    st.title("📜 Historique des Actions")
    st.subheader("Journal d'audit des modifications de la plateforme")

    col_clear, _ = st.columns([1, 3])
    with col_clear:
        if st.button("🗑️ Vider l'historique", type="primary"):
            st.session_state.confirm_clear_history = True
        if st.session_state.get("confirm_clear_history"):
            st.warning("Confirmer le vidage complet ?")
            if st.button("Confirmer", key="confirm_clear_history_btn"):
                run("DELETE FROM audit_logs")
                st.session_state.pop("confirm_clear_history", None)
                st.success("Historique vidé !")
                st.rerun()

    st.write("### 🔍 Filtrer le journal d'audit")
    c1, c2, c3 = st.columns(3)
    with c1:
        all_authors = [r["username"] for r in q("SELECT DISTINCT username FROM audit_logs ORDER BY username")]
        filter_authors = st.multiselect("Filtrer par Auteur", all_authors)
    with c2:
        all_actions = [r["action"] for r in q("SELECT DISTINCT action FROM audit_logs ORDER BY action")]
        filter_actions = st.multiselect("Filtrer par Action", all_actions)
    with c3:
        all_tables = [r["table_name"] for r in q("SELECT DISTINCT table_name FROM audit_logs ORDER BY table_name")]
        filter_tables = st.multiselect("Filtrer par Table", all_tables)

    sql = "SELECT * FROM audit_logs"
    params, where = [], []
    if filter_authors:
        where.append("username IN ({})".format(", ".join(["?"] * len(filter_authors))))
        params += filter_authors
    if filter_actions:
        where.append("action IN ({})".format(", ".join(["?"] * len(filter_actions))))
        params += filter_actions
    if filter_tables:
        where.append("table_name IN ({})".format(", ".join(["?"] * len(filter_tables))))
        params += filter_tables
    if where:
        sql += " WHERE " + " AND ".join(where)
    sql += " ORDER BY timestamp DESC"
    logs = q(sql, *params)

    st.write("---")
    if not logs:
        st.info("Aucun historique correspondant aux critères.")
    else:
        emojis = {"INSERT": "➕", "UPDATE": "📝", "UPSERT": "💾", "DELETE": "🗑️", "SCHEMA_ADD": "⚙️➕", "SCHEMA_DROP": "⚙️🗑️"}
        for log in logs:
            emoji = emojis.get(log["action"], "🔔")
            st.markdown(
                f"### {emoji} {log['action']} — Table `{log['table_name']}`\n"
                f"*   **Auteur** : `👤 {log['username']}`\n"
                f"*   **Référence** : `{log['record_id']}`\n"
                f"*   **Détails** : {log['details']}\n"
                f"*   **Date** : 📅 *{log['timestamp']}*"
            )
            st.write("---")
